# 01 · Data cleaning

**Owner:** Person 2 — Data Cleaning  
**Run from:** the `notebooks/` folder

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # so `src` can be imported from notebooks/
warnings.filterwarnings("ignore")

import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
FIG = Path.cwd().parent / "reports" / "figures"


## 1. Load the raw data

In [ ]:
from src.preprocess import load_raw, clean, add_features, remove_outliers

raw = load_raw()
print(raw.shape)
raw.head()

In [ ]:
raw.info()

## 2. Data quality checks

In [ ]:
print("Missing values:")
raw.isnull().sum()[lambda s: s > 0]

In [ ]:
raw.describe().T

In [ ]:
print(raw["Marital_Status"].value_counts(), "\n")
print(raw["Education"].value_counts())

In [ ]:
print("Constant columns:", [c for c in raw if raw[c].nunique() == 1])
print("Oldest birth years:", sorted(raw["Year_Birth"])[:5])
print("Highest incomes:", raw["Income"].nlargest(3).tolist())
print("Duplicate rows (ignoring ID):", raw.drop(columns="ID").duplicated().sum())

**Findings**
- `Income`: 24 missing → fill with median
- `Year_Birth` as early as 1893 → impossible ages, remove
- One `Income` of 666,666 → remove
- `Marital_Status` has `YOLO`, `Absurd`, `Alone` → treat as living alone
- `Z_CostContact`, `Z_Revenue` constant → drop
- 182 duplicate rows apart from ID → kept (TODO: team to decide)

## 3. Apply cleaning (code lives in `src/preprocess.py`)

In [ ]:
df = remove_outliers(add_features(clean(raw)))
print(raw.shape, "->", df.shape)
df.isnull().sum().sum()

## TODO
- [ ] Boxplots of Income and Age before/after outlier removal
- [ ] Decide on duplicate rows and document the decision